# Use guard clauses to flatten nested conditions

**The question.** You have a function whose real work is buried three or four `if` blocks deep because of the preconditions it has to check first. You want the main logic to sit at one indentation level and the preconditions to read as a checklist.

Guard clauses — `if <bad>: return …` (or `raise …`) at the top of the function — are the pattern. Each guard handles one invalid case and exits. Anything past the last guard can assume the inputs are good.

In [ ]:
# Mock objects so the example runs
class Customer:
    def __init__(self, logged_in): self.logged_in = logged_in
class Item:
    def __init__(self, price): self.price = price
class Order:
    def __init__(self, customer, cart, payment_method):
        self.customer = customer
        self.cart = cart
        self.payment_method = payment_method

valid = Order(Customer(True), [Item(9.99), Item(14.50)], 'Visa')


def process_order(order):
    # Guard clauses: handle every bad case up front, then proceed.
    if order is None:
        return 'Error: no order supplied'
    if not order.customer.logged_in:
        return 'Error: customer not logged in'
    if not order.cart:
        return 'Error: cart is empty'
    if not order.payment_method:
        return 'Error: no payment method'

    # The real work, at one indentation level
    total = sum(item.price for item in order.cart)
    return f'Charged £{total:.2f} via {order.payment_method}'


print(process_order(valid))
print(process_order(None))
print(process_order(Order(Customer(False), [Item(1)], 'Visa')))


In [ ]:
# Variant: guard with `raise`, not `return`
def process_order_raise(order):
    if order is None:
        raise ValueError('no order supplied')
    if not order.customer.logged_in:
        raise PermissionError('customer not logged in')
    if not order.cart:
        raise ValueError('cart is empty')
    if not order.payment_method:
        raise ValueError('no payment method')
    total = sum(item.price for item in order.cart)
    return f'Charged £{total:.2f} via {order.payment_method}'

print(process_order_raise(valid))
try:
    process_order_raise(None)
except ValueError as e:
    print(f'Caught: {e}')


In [ ]:
# Variant: guards as short-circuit dispatch
def describe(value):
    if value is None:
        return 'no value'
    if isinstance(value, bool):                 # must come before int!
        return 'a yes/no'
    if isinstance(value, (int, float)):
        return f'a number: {value}'
    if isinstance(value, str):
        return f'text of length {len(value)}'
    if isinstance(value, (list, tuple, set)):
        return f'a collection of {len(value)} item(s)'
    return f'some other thing: {type(value).__name__}'

for v in [None, True, 42, 3.14, 'hello', [1, 2, 3], {'a': 1}]:
    print(f'{v!r:15} -> {describe(v)}')


## Why it works

The guarded version reads as *"here are the things that would stop us, here is what we do otherwise."* The nested version forces the reader to assemble that mental model themselves — they have to track each open `if` in their head until they finally reach the real work at the bottom.

The win compounds with size. Add a fifth precondition to the nested version and you nest one level deeper (and every existing branch moves). Add it to the guarded version and you add one more `if`/`return` pair at the top. The main logic doesn't move at all.

Guards also remove a whole class of "redundant `else` after return" noise — once the early branch has returned, there's no need for the `else` that wraps the remainder.

## Trade-offs

**Return vs. raise.** Return a string (or a sentinel, or `None`) when the caller is going to branch on the result anyway. Raise when the invalid case is genuinely exceptional and the caller would have to check-and-raise itself. In application code, raising is usually the cleaner signal — the call site catches what it knows how to handle and lets the rest bubble up.

**Keep the nesting when:**

- The branches are doing genuinely different work, not handling errors — there are two real paths through the function.
- The conditions are interdependent in a way that flattening obscures. Sometimes `if A and B and C:` reads better than three separate guards.
- The function is tiny. For a three-line function, the nesting isn't hurting anyone.

**Watch the guard ordering.** For `isinstance` dispatch, remember that `bool` is a subclass of `int` — check `isinstance(value, bool)` before `isinstance(value, int)` or every boolean will be reported as a number.

## Related reading

- [Avoid common conditional mistakes](avoid-common-conditional-mistakes.md) — including the redundant-`else`-after-`return` pattern that guard clauses naturally eliminate.
- [Choose between if/elif chains, dict dispatch, and match/case](choose-between-conditional-patterns.ipynb) — when guards stop scaling and dispatch tables take over.
- [Truthiness rules](../reference/truthiness-rules.md) — what `if x:` actually tests for.
